# LangMem — Short-Term Memory via Message Summarization

## Why do we need summarization?

LLMs have a **context window limit** — they can only "see" a fixed number of tokens at once.  
In a long conversation, older messages eat up precious tokens, leaving less room for the current exchange.

**Message summarization** solves this by compressing old messages into a short summary, so the LLM still *knows what happened earlier* without having to read every message word-for-word.

---

## What is a "Running Summary"?

A `RunningSummary` is a compact object that holds:
- **`summary`** — a text description of what happened in the compressed messages
- **`summarized_message_ids`** — the IDs of messages that were compressed
- **`last_summarized_message_id`** — a pointer to the last message included in the summary

Each time new messages exceed the token budget, the running summary is **updated** (not replaced), so context accumulates incrementally.

---

## Three Approaches in This Notebook

| # | Style | Best For |
|---|-------|----------|
| 1 | `summarize_messages()` — inline function | Simple graphs where summarization lives inside the LLM node |
| 2 | `SummarizationNode` — dedicated graph node | Graphs where you want clean separation of concerns |
| 3 | `SummarizationNode` in a ReACT Agent | Tool-using agents where every reasoning loop needs fresh summarization |

---

## Token Parameters Explained

These appear in all 3 examples — understand them once, apply everywhere:

| Parameter | Meaning |
|-----------|---------|
| `max_tokens` | Max tokens allowed in the final message list passed to the LLM |
| `max_tokens_before_summary` | When total tokens exceed this, summarization kicks in |
| `max_summary_tokens` | How long the generated summary text can be |

---

# Approach 1 — `summarize_messages()` (Inline Function)

## How it works

`summarize_messages()` is a plain Python function you call **inside** your `call_model` node.  
It checks the current token count, and if it exceeds `max_tokens_before_summary`, it:
1. Calls the summarization model to compress older messages into a short text
2. Replaces those old messages with a single synthetic "summary" message
3. Returns the trimmed list along with an updated `RunningSummary`

The `RunningSummary` is stored directly in your graph state as `state["summary"]` — you manage it yourself.

## Graph structure

```
START → call_model
```

One node does everything: summarize (if needed) + call the LLM.  
Simple, but the node has two responsibilities mixed together.

In [12]:
# ─────────────────────────────────────────────────────────────────
# APPROACH 1: summarize_messages() — Inline Summarization Function
# ─────────────────────────────────────────────────────────────────
#
# How it works:
#   - summarize_messages() is called INSIDE the call_model node.
#   - It inspects the current message list and decides whether
#     the token count exceeds max_tokens_before_summary.
#   - If yes → it calls the summarization_model to compress older
#     messages into a RunningSummary, then returns the trimmed list.
#   - If no  → it returns messages unchanged (no summarization needed yet).
#   - The LLM then uses the (possibly trimmed) message list to respond.
#
# Graph structure:
#   START → call_model   (single node does everything)
#
# ─────────────────────────────────────────────────────────────────

from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langmem.short_term import summarize_messages, RunningSummary
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o")

# We use a weaker/cheaper model for summarization (bound to 128 output tokens).
# Summarization doesn't need to be creative — just concise.
summarization_model = model.bind(max_tokens=128)


# Extend MessagesState to also carry the running summary across turns.
# RunningSummary | None means it starts as None and gets populated after
# the first summarization event.
class SummaryState(MessagesState):
    summary: RunningSummary | None


def call_model(state: SummaryState) -> SummaryState:
    # ── Step 1: Summarize if needed ──────────────────────────────
    # summarize_messages() returns a SummarizationResult with two fields:
    #   .messages         → the trimmed (or unchanged) message list
    #   .running_summary  → updated RunningSummary (or None if no summarization happened)
    summarization_result = summarize_messages(
        state["messages"],

        # Pass the previously saved summary so it can be updated incrementally.
        # Without this, every call would re-summarize from scratch — wasteful!
        running_summary=state.get("summary"),

        # Tells summarize_messages how to count tokens for this specific model.
        token_counter=model.get_num_tokens_from_messages,

        # The model used to generate the summary text.
        model=summarization_model,

        # Hard cap: the final message list given to the LLM must fit in 256 tokens.
        max_tokens=256,

        # Summarization only triggers once we exceed 256 tokens. Below this, all
        # messages are passed as-is (no summarization overhead).
        max_tokens_before_summary=256,

        # The summary text itself cannot exceed 128 tokens.
        max_summary_tokens=128,
    )

    # ── Step 2: Call the main LLM with the trimmed messages ──────
    response = model.invoke(summarization_result.messages)

    # ── Step 3: Build state update ───────────────────────────────
    state_update = {"messages": [response]}

    # Only save the summary back to state if summarization actually happened.
    # If running_summary is None here, the conversation was short enough that
    # no compression was needed yet — no need to update state.
    if summarization_result.running_summary:
        state_update["summary"] = summarization_result.running_summary

    return state_update


checkpointer = InMemorySaver()  # Persists state across graph.invoke() calls
builder = StateGraph(SummaryState)
builder.add_node(call_model)
builder.add_edge(START, "call_model")
graph = builder.compile(checkpointer=checkpointer)

# ── Test the graph across 4 turns ────────────────────────────────
# Turn 1 — introduce name (will eventually be summarized away)
# Turn 2 & 3 — long poem responses push token count past the limit
# Turn 4 — asks about the name; the model has forgotten it (it was compressed!)
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": "hi, my name is bob"}, config)
graph.invoke({"messages": "write a short poem about cats"}, config)
graph.invoke({"messages": "now do the same but for dogs"}, config)
graph.invoke({"messages": "what's my name?"}, config)
# ⚠️  Expected: The model will say it doesn't know the name.
# Why? The "hi, my name is bob" message was compressed into the summary,
# but the summary text ("poems about cats and dogs") didn't retain the name.

{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='3a68dddf-77f7-4c42-8819-eca97b459e35'),
  AIMessage(content='Hello Bob! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_e6c36a96dd', 'id': 'chatcmpl-DMRuAMjh0RTcxS655Pdcwl0zFtHEy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d191d-748b-7711-86f7-af9813e1382f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu

# Approach 2 — `SummarizationNode` (Dedicated Graph Node)

## How it differs from Approach 1

In Approach 1, summarization was **baked into** the `call_model` node — the node did two jobs at once (summarize + call LLM).

Here, summarization is extracted into its **own dedicated node** (`SummarizationNode`).  
The graph becomes: `START → summarize → call_model`

### Key design difference: Private Input State

`SummarizationNode` outputs its result into a **private key** (`summarized_messages`) inside a `LLMInputState` TypedDict.  
The `call_model` node only receives `LLMInputState` — it never sees the raw `messages` list or the checkpointed `context`.

This is called **state isolation** — the summarization output is only visible to the node that needs it, keeping the rest of the graph's state clean.

### Where does the `RunningSummary` live?

Unlike Approach 1 (where `summary` was a top-level state field), here the running summary is stored in `context["running_summary"]`.  
`SummarizationNode` manages reading and writing to `context` automatically — you don't need to wire it manually.

### Observation from the output

The final message shows `"what's my name?"` returns *"I don't know your name"* — same bug as Approach 1.  
This is because the first message (`"hi, my name is bob"`) got compressed into the summary, but the **summary text** didn't explicitly capture the name.  
This is a known limitation of summarization — lossy compression can drop details.

In [13]:
# ─────────────────────────────────────────────────────────────────
# APPROACH 2: SummarizationNode — Dedicated Graph Node
# ─────────────────────────────────────────────────────────────────
#
# How it works:
#   - Summarization is pulled OUT of call_model and into its own node.
#   - The graph runs: START → summarize → call_model
#   - SummarizationNode reads from `messages` + `context` (state),
#     produces `summarized_messages` (a private key only visible to call_model),
#     and writes the updated RunningSummary back into `context`.
#
# Key advantage over Approach 1:
#   - Single Responsibility Principle: each node has one job.
#   - call_model is now a pure function: receives messages, returns response.
#   - Easier to test, swap, or extend either node independently.
#
# ─────────────────────────────────────────────────────────────────

from typing import Any, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langmem.short_term import SummarizationNode, RunningSummary

model = ChatOpenAI(model="gpt-4o")
summarization_model = model.bind(max_tokens=128)


# The full graph state — stored in the checkpointer between turns.
# `context` is a catch-all dict; SummarizationNode will write
# context["running_summary"] = RunningSummary(...) into it automatically.
class State(MessagesState):
    context: dict[str, Any]


# ── Private input state for call_model ───────────────────────────
# This TypedDict is NOT the full graph state — it's a "view" that
# LangGraph passes to call_model. It contains only the data that
# call_model actually needs. The raw `messages` and `context` fields
# are NOT exposed here, enforcing a clean separation.
#
# `summarized_messages` is populated by SummarizationNode right before
# call_model runs. It contains either:
#   - The full message list (if no summarization was needed), OR
#   - A synthetic "summary" message followed by only the recent messages
class LLMInputState(TypedDict):
    summarized_messages: list[AnyMessage]
    context: dict[str, Any]


# ── SummarizationNode setup ───────────────────────────────────────
# SummarizationNode is a drop-in LangGraph node. Under the hood it:
#   1. Reads state["messages"] and state["context"]["running_summary"]
#   2. Counts tokens; if over budget, calls `model` to compress old messages
#   3. Writes the trimmed list to the private `summarized_messages` key
#   4. Updates state["context"]["running_summary"] with the new RunningSummary
summarization_node = SummarizationNode(
    token_counter=model.get_num_tokens_from_messages,
    model=summarization_model,
    max_tokens=256,               # Budget for the final message list
    max_tokens_before_summary=256,  # Summarize once this threshold is crossed
    max_summary_tokens=128,       # Keep the summary itself short
)


# call_model only receives LLMInputState — it has no idea whether
# summarization happened or not. It just uses whatever messages it gets.
def call_model(state: LLMInputState):
    response = model.invoke(state["summarized_messages"])
    return {"messages": [response]}


checkpointer = InMemorySaver()
builder = StateGraph(State)
builder.add_node(call_model)
builder.add_node("summarize", summarization_node)  # <── new dedicated node

# Execution order: summarize first, then call the LLM
builder.add_edge(START, "summarize")
builder.add_edge("summarize", "call_model")
graph = builder.compile(checkpointer=checkpointer)

# ── Test: same 4-turn conversation as Approach 1 ─────────────────
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": "hi, my name is bob"}, config)
graph.invoke({"messages": "write a short poem about cats"}, config)
graph.invoke({"messages": "now do the same but for dogs"}, config)
graph.invoke({"messages": "what's my name?"}, config)
# ⚠️  Same result as Approach 1: name is forgotten after summarization.
# The behaviour is identical — only the code structure is different.

{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='6b648a28-d9aa-4053-906d-d22c64aa2a16'),
  AIMessage(content='Hello Bob! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_e6c36a96dd', 'id': 'chatcmpl-DMRuGpmbPJ36WHr8veC1NgYFrbTrJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d191d-8c5c-7252-bb98-463ce957360d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu

In [14]:
# ── Approach 2: Inspect the state after running ───────────────────
#
# ⚠️  IMPORTANT — there are TWO different message lists. Don't confuse them:
#
#   state.values["messages"]
#     → The FULL checkpoint history. Every message ever sent is stored here
#       by InMemorySaver. This is the complete archive.
#       The LLM does NOT see this directly.
#
#   summarized_messages  (private key, only exists during graph execution)
#     → What the LLM ACTUALLY sees. Produced by SummarizationNode each turn.
#       Contains: [synthetic summary message] + [recent messages that fit in budget]
#       This is ephemeral — it is NOT persisted in the checkpointer, so you
#       cannot inspect it after the fact.
#
# This is why printing state.values["messages"] can be misleading —
# it shows "hi, my name is bob" as if it's still in context, but the LLM
# actually received the compressed summary instead.

state = graph.get_state({"configurable": {"thread_id": "1"}})

print("=== Running Summary (what replaced the old messages) ===")
running_summary = state.values.get("context", {}).get("running_summary")
if running_summary:
    print(f"  Text: {running_summary.summary}")
    print(f"  # of compressed messages: {len(running_summary.summarized_message_ids)}")
else:
    print("  No summarization has occurred yet.")
print()

print("=== Full checkpoint history (NOT what the LLM sees) ===")
for m in state.values["messages"]:
    marker = " ← compressed into summary" if (
        running_summary and m.id in running_summary.summarized_message_ids
    ) else ""
    print(f"  [{m.type:8s}]: {str(m.content)[:80]}{marker}")

print()
print("=== What the LLM actually sees (reconstructed) ===")
print("  [summary ]: " + (running_summary.summary if running_summary else "(none)"))
for m in state.values["messages"]:
    if running_summary and m.id in running_summary.summarized_message_ids:
        continue  # These were compressed — LLM doesn't see them individually
    print(f"  [{m.type:8s}]: {str(m.content)[:80]}")

# ── Why the name is forgotten ─────────────────────────────────────
# "hi, my name is bob" gets compressed into the running summary.
# The summary model writes: "The conversation involved poems about cats and dogs."
# The name "bob" is never mentioned in the summary → the LLM has no way to recall it.
#
# Fix options:
#   1. Use long-term memory (create_memory_store_manager) to extract and
#      persist facts like names separately from the message list.
#   2. Provide a custom summarization prompt that explicitly says:
#      "Always preserve names, dates, and user-stated preferences."

Deserializing unregistered type langmem.short_term.summarization.RunningSummary from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('langmem.short_term.summarization', 'RunningSummary')]


=== Running Summary (what replaced the old messages) ===
  Text: The conversation involves a request to create a poem about dogs, followed by a response that presents a poem celebrating their playful spirit, loyalty, and companionship.
  # of compressed messages: 6

=== Full checkpoint history (NOT what the LLM sees) ===
  [human   ]: hi, my name is bob ← compressed into summary
  [ai      ]: Hello Bob! How can I assist you today? ← compressed into summary
  [human   ]: write a short poem about cats ← compressed into summary
  [ai      ]: In sunlit patches, cats recline,  
With eyes like jewels, they softly shine.  
W ← compressed into summary
  [human   ]: now do the same but for dogs ← compressed into summary
  [ai      ]: In morning's light, dogs bound with glee,  
With wagging tails, they run so free ← compressed into summary
  [human   ]: what's my name?
  [ai      ]: I'm sorry, I don't have access to personal information, so I don't know your nam

=== What the LLM actually sees (

# Fix — Custom Summarization Prompts to Preserve Names & Facts

## The problem

The default summarization prompt simply says:
> *"Create a summary of the conversation above."*

This gives the model no guidance on what to preserve. Facts like names, dates, and preferences get dropped silently.

## The fix

LangMem lets you replace the default prompts with your own `ChatPromptTemplate` objects via two parameters:

| Parameter | When it's used | Template variables |
|-----------|---------------|-------------------|
| `initial_summary_prompt` | First time a summary is created (no prior summary exists) | `{messages}` |
| `existing_summary_prompt` | Updating an existing summary with new messages | `{messages}`, `{existing_summary}` |

You only need to change the **user instruction text** inside each template — the `{messages}` and `{existing_summary}` placeholders must remain exactly as-is (LangMem fills them in automatically).

## Works for both `summarize_messages()` and `SummarizationNode`

Both accept the same two parameters, so the fix applies to all 3 approaches.

In [15]:
from typing import Any, TypedDict

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langmem.short_term import SummarizationNode, RunningSummary

model = ChatOpenAI(model="gpt-4o")
summarization_model = model.bind(max_tokens=256)  # was 128 — too tight for fact-preserving summaries

# ── Custom prompts that tell the summarizer WHAT to preserve ──────
#
# The {messages} and {existing_summary} placeholders are REQUIRED —
# LangMem substitutes them at runtime. Only change the instruction text.

# Used the FIRST time a summary is created (no existing summary yet).
initial_summary_prompt = ChatPromptTemplate.from_messages(
    [
        ("placeholder", "{messages}"),   # LangMem fills this with the conversation
        (
            "user",
            "Create a concise summary of the conversation above.\n"
            "You MUST explicitly preserve the following if they appear:\n"
            "  - People's names (e.g. 'The user's name is Bob')\n"
            "  - Dates or times mentioned\n"
            "  - Any preferences or facts the user stated about themselves\n"
            "Do not omit these details even if the conversation is mostly about other topics.",
        ),
    ]
)

# Used when UPDATING an existing summary with newer messages.
# {existing_summary} contains the previously generated summary text.
existing_summary_prompt = ChatPromptTemplate.from_messages(
    [
        ("placeholder", "{messages}"),   # LangMem fills this with the new messages
        (
            "user",
            "Current summary: {existing_summary}\n\n"
            "Update the summary to include the new messages above.\n"
            "You MUST continue to preserve:\n"
            "  - People's names\n"
            "  - Dates or times\n"
            "  - Any preferences or facts the user stated about themselves\n"
            "Do not drop these details when rewriting the summary.",
        ),
    ]
)

# ── Why the token budget matters ──────────────────────────────────
#
# The relationship between the three token params:
#
#   max_tokens              = total budget for (summary message + live messages)
#   max_summary_tokens      = cap on how long the summary text can be
#   live messages budget    = max_tokens - actual_summary_tokens
#
# With max_summary_tokens=128 (original):
#   - Our custom instruction text is ~50 tokens
#   - The model only has ~78 tokens left for actual summary content
#   - Not enough to capture name + poems + context → name gets dropped
#
# Fix: increase max_summary_tokens to 256 and max_tokens to 1024
#   - Summary can now be ~256 tokens: plenty to say "User is Bob, asked for poems..."
#   - Live messages get the remaining ~768 tokens of context
#   - max_tokens_before_summary=1024 means compression only kicks in for long conversations

class State(MessagesState):
    context: dict[str, Any]

class LLMInputState(TypedDict):
    summarized_messages: list[AnyMessage]
    context: dict[str, Any]

summarization_node = SummarizationNode(
    token_counter=model.get_num_tokens_from_messages,
    model=summarization_model,
    max_tokens=1024,                  # total budget: summary + live messages
    max_tokens_before_summary=1024,   # only compress once conversation exceeds this
    max_summary_tokens=256,           # was 128 — now enough room to preserve facts
    initial_summary_prompt=initial_summary_prompt,
    existing_summary_prompt=existing_summary_prompt,
)

def call_model(state: LLMInputState):
    response = model.invoke(state["summarized_messages"])
    return {"messages": [response]}

checkpointer = InMemorySaver()
builder = StateGraph(State)
builder.add_node(call_model)
builder.add_node("summarize", summarization_node)
builder.add_edge(START, "summarize")
builder.add_edge("summarize", "call_model")
graph_fixed = builder.compile(checkpointer=checkpointer)

# ── Same 4-turn test ──────────────────────────────────────────────
config = {"configurable": {"thread_id": "fix-2"}}
graph_fixed.invoke({"messages": "hi, my name is bob"}, config)
graph_fixed.invoke({"messages": "write a short poem about cats"}, config)
graph_fixed.invoke({"messages": "now do the same but for dogs"}, config)
result = graph_fixed.invoke({"messages": "what's my name?"}, config)

print("=== LLM response to 'what's my name?' ===")
print(result["messages"][-1].content)
print()

state = graph_fixed.get_state(config)
running_summary = state.values.get("context", {}).get("running_summary")
print("=== Running Summary (should now mention 'Bob') ===")
if running_summary:
    print(running_summary.summary)
else:
    print("No summarization triggered — conversation still within token budget.")

=== LLM response to 'what's my name?' ===
Your name is Bob.

=== Running Summary (should now mention 'Bob') ===
No summarization triggered — conversation still within token budget.


# Approach 3 — `SummarizationNode` in a ReACT Agent (Tool-Using Agent)

## What is a ReACT Agent?

ReACT = **Re**asoning + **Act**ing.  
The agent loops between:
1. **Reasoning** — the LLM decides what to do next
2. **Acting** — calling a tool to get real-world information
3. **Observing** — reading the tool's response and reasoning again

This loop continues until the LLM decides it has enough information to give a final answer.

## Why does a ReACT agent need summarization more urgently?

Each iteration of the loop adds messages: a tool-call message + a tool-response message.  
After 5–10 tool calls, the context fills up fast — especially with verbose tool outputs.  
Summarization keeps the context window manageable across many reasoning steps.

## The critical wiring difference

In Approach 2, the graph was linear: `START → summarize → call_model`  

In Approach 3, after `tools` executes, control loops back to `summarize_node` — **not** directly to `call_model`.

```
START → summarize_node → call_model → [tools → summarize_node → call_model → ...]
                                    ↘ END
```

This means every time the agent prepares to think again (after a tool call), it **re-summarizes** first.  
The LLM always starts its reasoning with a fresh, token-efficient context.

## `max_tokens_before_summary` is larger here

Notice `max_tokens_before_summary=1024` (vs 256 in the earlier examples).  
Tool responses can be large — you want to give the agent more room before compressing,  
otherwise a single tool response could trigger summarization immediately.

In [16]:
# ─────────────────────────────────────────────────────────────────
# APPROACH 3: SummarizationNode in a ReACT (Tool-Using) Agent
# ─────────────────────────────────────────────────────────────────
#
# How it works:
#   - Same SummarizationNode as Approach 2, but now embedded in a
#     multi-step reasoning loop that can call external tools.
#
# Graph wiring:
#   START
#     ↓
#   summarize_node           ← always runs first (and after every tool call)
#     ↓
#   call_model               ← decides: answer now, or call a tool?
#     ↓              ↘
#   tools            END     ← if tool_calls present → run tools
#     ↓
#   (back to summarize_node) ← re-summarize before next reasoning step
#
# Why loop back to summarize_node after tools?
#   Tool responses add messages to state. Without re-summarizing, the context
#   grows with each tool call. By routing tools → summarize_node, we ensure
#   the LLM always starts a reasoning step with a lean context.
#
# ─────────────────────────────────────────────────────────────────

from typing import Any, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langmem.short_term import SummarizationNode, RunningSummary


class State(MessagesState):
    context: dict[str, Any]  # Holds running_summary, managed by SummarizationNode


# ── Fake search tool (simulates a real web search) ───────────────
# In a real app this would call a search API. Here we return hardcoded
# responses so we can test the agent without external dependencies.
def search(query: str):
    """Search the web."""
    if "weather" in query.lower():
        return "The weather is sunny in New York, with a high of 104 degrees."
    elif "broadway" in query.lower():
        return "Hamilton is always on!"
    else:
        raise "Not enough information"

tools = [search]

model = ChatOpenAI(model="gpt-4o")
summarization_model = model.bind(max_tokens=128)

# ── SummarizationNode ─────────────────────────────────────────────
# max_tokens_before_summary is set higher (1024) compared to earlier examples.
# Tool responses can be verbose — we give the agent more breathing room
# before forcing a compression, to avoid summarizing immediately after
# the very first tool call.
summarization_node = SummarizationNode(
    token_counter=model.get_num_tokens_from_messages,
    model=summarization_model,
    max_tokens=256,
    max_tokens_before_summary=1024,  # Higher threshold for tool-heavy conversations
    max_summary_tokens=128,
)


# Private input state — same pattern as Approach 2.
# call_model only sees summarized_messages, not raw state.
class LLMInputState(TypedDict):
    summarized_messages: list[AnyMessage]
    context: dict[str, Any]


def call_model(state: LLMInputState):
    # bind_tools() tells the LLM which tools it can call.
    # The LLM may return a message with tool_calls=[...] instead of a text answer.
    response = model.bind_tools(tools).invoke(state["summarized_messages"])
    return {"messages": [response]}


# ── Router: continue looping or exit? ────────────────────────────
# After call_model runs, this function checks the last message.
# If the LLM included tool_calls → route to "tools" node.
# If the LLM gave a plain text answer → we're done, route to END.
def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return END       # No tool calls → LLM has final answer
    else:
        return "tools"   # Tool calls present → execute them


checkpointer = InMemorySaver()
builder = StateGraph(State)
builder.add_node("summarize_node", summarization_node)
builder.add_node("call_model", call_model)
builder.add_node("tools", ToolNode(tools))  # ToolNode auto-executes tool_calls

# Entry point is the summarize node (not call_model directly!)
builder.set_entry_point("summarize_node")
builder.add_edge("summarize_node", "call_model")

# After call_model: either go to tools or END
builder.add_conditional_edges("call_model", should_continue, path_map=["tools", END])

# ── The critical loop-back edge ───────────────────────────────────
# After tools run, route back to summarize_node (not call_model).
# This ensures: new tool messages get compressed BEFORE the LLM reasons again.
builder.add_edge("tools", "summarize_node")

graph = builder.compile(checkpointer=checkpointer)

# ── Test across 3 turns ───────────────────────────────────────────
# Turn 1: Simple greeting — no tool call needed
# Turn 2: Weather query — agent calls search("NYC weather"), gets result, answers
# Turn 3: Broadway query — agent calls search("Broadway"), gets result, answers
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": "hi, i am bob"}, config)
graph.invoke({"messages": "what's the weather in nyc this weekend"}, config)
graph.invoke({"messages": "what's new on broadway?"}, config)

{'messages': [HumanMessage(content='hi, i am bob', additional_kwargs={}, response_metadata={}, id='4a49d26f-0b99-4bd5-ae2a-3df30ac727ec'),
  AIMessage(content='Hello Bob! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 45, 'total_tokens': 56, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_302b87bc9f', 'id': 'chatcmpl-DMRuRgEA917zMSOpl9lzN1AiAQCvf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d191d-ba66-7021-b00e-fd8c70bbcc69-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 11, 'total_tokens': 56, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_toke

# ── Summary & Comparison of All 3 Approaches ─────────────────────────────────────

## Side-by-Side Comparison

| | Approach 1 | Approach 2 | Approach 3 |
|---|---|---|---|
| **Tool** | `summarize_messages()` | `SummarizationNode` | `SummarizationNode` |
| **Where summarization runs** | Inside `call_model` node | Dedicated `summarize` node | Dedicated `summarize_node` |
| **Graph structure** | `START → call_model` | `START → summarize → call_model` | `START → summarize ⇄ call_model ⇄ tools` |
| **Summary stored in** | `state["summary"]` (explicit) | `state["context"]["running_summary"]` (auto) | `state["context"]["running_summary"]` (auto) |
| **State isolation** | No — all in one node | Yes — `LLMInputState` hides raw messages | Yes — same as Approach 2 |
| **Supports tools?** | No | No | Yes |
| **Code complexity** | Low | Medium | Medium-High |
| **Best for** | Quick prototypes | Clean production graphs | Tool-using agents |

---

## When to use which

- **Approach 1** — You're prototyping and want minimal boilerplate. The graph is simple (no tools).  
- **Approach 2** — You want clean architecture. You may add more nodes later and don't want summarization tangled with LLM logic.  
- **Approach 3** — Your agent uses tools. You need summarization at every reasoning step to prevent context blowup across tool-call loops.

---

## The "lossy compression" gotcha

All 3 approaches demonstrate the same limitation: if an important detail (e.g., the user's name) appears early in the conversation and later gets compressed, **the summary may not preserve it**.

**Fix strategies:**
1. Store important facts in long-term memory (LangMem's `create_memory_store_manager`) — separate from the message list
2. Increase `max_tokens_before_summary` to delay compression
3. Use a higher-quality (but more expensive) summarization model
4. Provide a custom summarization prompt that instructs the model to preserve specific fact types